# Sprint 2: Spatial Questions to Market Data

**GeoRAD Emerging Spatial Professionals Mentorship Program · Week 3**

This notebook helps you:

1. Upload and **inspect** your five-town GeoJSON from Sprint 1
2. Extract town names as workflow inputs
3. Run an **AI-assisted Google search** (via SerpApi Google AI Overview) for current land price signals
4. Build a structured **market dataset** for the next sprint

You do **not** need to build an AI model. Focus on understanding the workflow, running it, inspecting results, and documenting what you find.

> Keep your SerpApi key private. Do not commit this notebook or your key to a public GitHub repo.


## 0. Install packages

Run once per Colab session.


In [1]:
# SerpApi client + helpers for tables / files
%pip install -q google-search-results pandas


  Preparing metadata (setup.py) ... done


## 1. Configure your SerpApi key

1. Create a key at [serpapi.com](https://serpapi.com/)
2. Prefer Colab Secrets (`SERPAPI_API_KEY`) or paste when prompted

Never hard-code the key into a cell you will share publicly.


In [3]:
import os
from getpass import getpass

try:
    from google.colab import userdata  # type: ignore
    SERPAPI_API_KEY = userdata.get("SERPAPI_API_KEY")
except Exception:
    SERPAPI_API_KEY = None

if not SERPAPI_API_KEY:
    SERPAPI_API_KEY = os.environ.get("SERPAPI_API_KEY") or getpass("SerpApi API key: ").strip()

assert SERPAPI_API_KEY, "SerpApi API key is required."
print("SerpApi key loaded.")


SerpApi key loaded.


## 2. Upload your towns GeoJSON

Upload the **selected towns** GeoJSON from Sprint 1 (five towns inside your assigned LCDA).

GeoRAD town exports typically look like:

```json
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "ward": "Idiya",
        "lga": "Abeokuta North",
        "state": "Ogun"
      },
      "geometry": { "type": "MultiPolygon", "coordinates": [] }
    }
  ]
}
```

**Town name field:** use `ward` (primary). The notebook will detect this automatically.


In [11]:
from pathlib import Path
import json

try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    GEOJSON_PATH = next(iter(uploaded))
except Exception:
    GEOJSON_PATH = "sample_towns.geojson"
    print(f"Using local path: {GEOJSON_PATH}")

path = Path(GEOJSON_PATH)
assert path.exists(), f"File not found: {path}"
raw = path.read_text(encoding="utf-8")
geojson = json.loads(raw)
print(f"Loaded: {path.name} ({path.stat().st_size:,} bytes)")


Saving Igando_Ikotun.geojson to Igando_Ikotun (1).geojson
Loaded: Igando_Ikotun (1).geojson (18,904 bytes)


## 3. Inspect the GeoJSON

Before searching the market, confirm the file is valid and that the **ward** names look correct.


In [28]:
from collections import Counter

# Preferred property keys for the place name, in order.
# Mentorship town GeoJSONs from GeoRAD use `ward`.
NAME_KEYS = [
    "ward",
    "Ward",
    "ward_name",
    "name",
    "Name",
    "town",
    "Town",
    "Ward",
    "Ward",
    "label",
    "Label",
]

# Optional manual override, e.g. TOWN_NAME_FIELD = "ward"
TOWN_NAME_FIELD = "wardname"


def detect_name_field(features_list) -> str | None:
    if TOWN_NAME_FIELD:
        return TOWN_NAME_FIELD
    for key in NAME_KEYS:
        values = []
        for feature in features_list:
            props = feature.get("properties") or {}
            if not isinstance(props, dict):
                continue
            value = props.get(key)
            if value is not None and str(value).strip():
                values.append(str(value).strip())
        if len(values) == len(features_list):
            return key
    return None


def feature_name(props: dict, name_field: str | None):
    if not isinstance(props, dict) or not name_field:
        return None
    value = props.get(name_field)
    if value is None or not str(value).strip():
        return None
    return str(value).strip()


def geometry_summary(geom):
    if not geom or not isinstance(geom, dict):
        return "missing"
    gtype = geom.get("type", "unknown")
    coords = geom.get("coordinates")
    return f"{gtype} (coords present={coords is not None})"


print("=== Top-level ===")
print("type:", geojson.get("type"))
print("name:", geojson.get("name"))
print("crs :", geojson.get("crs"))
print("keys:", sorted(geojson.keys()))

features = geojson.get("features")
if not isinstance(features, list):
    raise TypeError("Expected a FeatureCollection with a features array.")

print("\n=== Features ===")
print("count:", len(features))
geom_types = Counter((f.get("geometry") or {}).get("type", "missing") for f in features)
print("geometry types:", dict(geom_types))

prop_keys = Counter()
for f in features:
    props = f.get("properties") or {}
    if isinstance(props, dict):
        prop_keys.update(props.keys())
print("property keys seen:", dict(prop_keys))

TOWN_NAME_FIELD = detect_name_field(features)
print("\nDetected town-name field:", TOWN_NAME_FIELD)
if not TOWN_NAME_FIELD:
    raise ValueError(
        "Could not detect a town-name property. "
        "Set TOWN_NAME_FIELD = 'ward' (or your field name) and re-run this cell."
    )

print("\n=== Feature preview ===")
for i, feature in enumerate(features, start=1):
    props = feature.get("properties") or {}
    name = feature_name(props, TOWN_NAME_FIELD)
    lga = props.get("lga") if isinstance(props, dict) else None
    state = props.get("state") if isinstance(props, dict) else None
    extra = []
    if lga:
        extra.append(f"lga={lga}")
    if state:
        extra.append(f"state={state}")
    suffix = f" | {', '.join(extra)}" if extra else ""
    print(f"{i}. {name or '(unnamed)'} | {geometry_summary(feature.get('geometry'))}{suffix}")

if len(features) != 5:
    print(f"\nWARNING: expected 5 towns, found {len(features)}. Continue only if intentional.")
else:
    print("\nOK: found 5 features.")

=== Top-level ===
type: FeatureCollection
name: Igando_Ikotun
crs : {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}
keys: ['crs', 'features', 'name', 'type']

=== Features ===
count: 5
geometry types: {'MultiPolygon': 5}
property keys seen: {'FID': 5, 'globalid': 5, 'uniq_id': 5, 'timestamp': 5, 'editor': 5, 'wardname': 5, 'wardcode': 5, 'lganame': 5, 'lgacode': 5, 'statename': 5, 'statecode': 5, 'amapcode': 5, 'status': 5, 'source': 5, 'urban': 5}

Detected town-name field: wardname

=== Feature preview ===
1. Igando | MultiPolygon (coords present=True)
2. Ikotun | MultiPolygon (coords present=True)
3. Isheri-Olofin | MultiPolygon (coords present=True)
4. Abaranje / Okerube | MultiPolygon (coords present=True)
5. Egan | MultiPolygon (coords present=True)

OK: found 5 features.


## 4. Extract the five town names

Uses the detected property (usually **`ward`**) as the search input.


In [29]:
towns = []
for feature in features:
    name = feature_name(feature.get("properties") or {}, TOWN_NAME_FIELD)
    if name:
        towns.append(name)

seen = set()
deduped = []
for town in towns:
    key = town.lower()
    if key in seen:
        continue
    seen.add(key)
    deduped.append(town)
towns = deduped

print(f"Using property: {TOWN_NAME_FIELD}")
print("Towns to search:")
for i, town in enumerate(towns, start=1):
    print(f"  {i}. {town}")

if len(towns) < 1:
    raise ValueError("No town names found. Check TOWN_NAME_FIELD / your GeoJSON.")
if len(towns) > 5:
    print("NOTE: more than 5 names found. Using the first 5.")
    towns = towns[:5]


Using property: wardname
Towns to search:
  1. Igando
  2. Ikotun
  3. Isheri-Olofin
  4. Abaranje / Okerube
  5. Egan


## 5. Market question template

Starting question (edit if you want to experiment later):

> What is the current land price per square metre in [Town], Lagos?


In [31]:
QUESTION_TEMPLATE = "What is the current land price per square metre in {town}, Lagos State?"

# Optional override for experimentation:
# QUESTION_TEMPLATE = "Current asking price of residential land per sqm in {town}, Lagos Nigeria"

for town in towns:
    print(QUESTION_TEMPLATE.format(town=town))


What is the current land price per square metre in Igando, Lagos State?
What is the current land price per square metre in Ikotun, Lagos State?
What is the current land price per square metre in Isheri-Olofin, Lagos State?
What is the current land price per square metre in Abaranje / Okerube, Lagos State?
What is the current land price per square metre in Egan, Lagos State?


## 6. Fetch Google AI Overview via SerpApi

Flow:

1. Google Search (`engine=google`) for the question (`hl=en`, `gl=ng`)
2. If AI Overview returns a `page_token`, call `engine=google_ai_overview` immediately
3. Collect overview text + first supporting reference

Run towns **sequentially**. AI Overview tokens expire quickly.


In [36]:
!pip install -q google-search-results

from datetime import date
import re
import time
from serpapi import GoogleSearch


def flatten_text_blocks(blocks) -> str:
    parts = []
    if not isinstance(blocks, list):
        return ""
    for block in blocks:
        if not isinstance(block, dict):
            continue
        if block.get("snippet"):
            parts.append(str(block["snippet"]))
        if block.get("list"):
            for item in block["list"]:
                if isinstance(item, dict) and item.get("snippet"):
                    parts.append(str(item["snippet"]))
                elif isinstance(item, str):
                    parts.append(item)
        if block.get("text_blocks"):
            parts.append(flatten_text_blocks(block["text_blocks"]))
    return "\n".join(p for p in parts if p)


def fetch_ai_overview(query: str) -> dict:
    search = GoogleSearch(
        {
            "engine": "google",
            "q": query,
            "hl": "en",
            "gl": "ng",
            "api_key": SERPAPI_API_KEY,
            "no_cache": "true",
        }
    )
    results = search.get_dict()
    if results.get("error"):
        return {"error": results["error"], "query": query}

    ai = results.get("ai_overview") or {}
    if ai.get("page_token"):
        follow = GoogleSearch(
            {
                "engine": "google_ai_overview",
                "page_token": ai["page_token"],
                "api_key": SERPAPI_API_KEY,
                "no_cache": "true",
            }
        )
        follow_results = follow.get_dict()
        if follow_results.get("error"):
            return {"error": follow_results["error"], "query": query, "stage": "ai_overview"}
        ai = follow_results.get("ai_overview") or follow_results

    text = flatten_text_blocks(ai.get("text_blocks"))
    if not text and ai.get("snippet"):
        text = str(ai["snippet"])

    references = ai.get("references") or []
    return {
        "query": query,
        "text": text.strip(),
        "references": references if isinstance(references, list) else [],
        "has_ai_overview": bool(text or references),
        "raw_keys": sorted(ai.keys()) if isinstance(ai, dict) else [],
    }


def extract_price_ngn_per_sqm(text: str):
    if not text:
        return None
    patterns = [
        r"₦\s*([\d,]+(?:\.\d+)?)\s*(?:per|/)?\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"NGN\s*([\d,]+(?:\.\d+)?)\s*(?:per|/)?\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"([\d,]+(?:\.\d+)?)\s*(?:naira|NGN|₦)\s*(?:per|/)\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"([\d,]+(?:\.\d+)?)\s*/\s*(?:sqm|sq\.?\s*m)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return f"₦{match.group(1).replace(',', '')}/sqm"
    match = re.search(r"₦\s*([\d,]+(?:\.\d+)?)", text)
    if match:
        return f"₦{match.group(1).replace(',', '')} (unit unclear; verify)"
    return None


def first_source(references: list) -> str:
    if not references:
        return ""
    ref = references[0] if isinstance(references[0], dict) else {}
    title = str(ref.get("title") or "").strip()
    link = str(ref.get("link") or ref.get("source") or "").strip()
    if title and link:
        return f"{title} | {link}"
    return link or title or ""


search_date = date.today().isoformat()
overview_cache = {}

for town in towns:
    query = QUESTION_TEMPLATE.format(town=town)
    print(f"\n=== {town} ===")
    print("Query:", query)
    result = fetch_ai_overview(query)
    overview_cache[town] = result
    if result.get("error"):
        print("ERROR:", result["error"])
    else:
        print("AI Overview found:", result.get("has_ai_overview"))
        preview = (result.get("text") or "")[:400]
        suffix = "..." if len(result.get("text") or "") > 400 else ""
        print("Text preview:", preview + suffix)
        print("References:", len(result.get("references") or []))
    time.sleep(1.5)

print("\nDone fetching AI Overviews.")


=== Igando ===
Query: What is the current land price per square metre in Igando, Lagos State?
AI Overview found: True
Text preview: The current land price in Igando, Lagos State , is approximately ₦85,380 per square metre, based on recent listings in the area (such as a 644.182 sqm plot on Taiwo Street listed for ₦55 million).
Price Breakdown & Examples
Taiwo Street, Igando : Taiwo Taiwo & Associates Listing — ₦55,000,000 for 644.18 sqm (~₦85,380/sqm)
General Range: Depending on the exact micro-location (e.g., closeness to the...
References: 2

=== Ikotun ===
Query: What is the current land price per square metre in Ikotun, Lagos State?
AI Overview found: True
Text preview: Land in Ikotun, Lagos State typically costs between ₦40,000 and ₦140,000 per square metre, depending on the exact location, proximity to major roads, and title documents.
Price Breakdown by Location & Type
Gated Estates / Developing Periphery: Plots (around 500 sqm) sell for about ₦20,000,000 outright, averaging ro

## 7. Build the Sprint 2 market dataset

Required columns:

| Field | Description |
| --- | --- |
| Town | Assigned town name |
| Current Land Price/sqm | Indicative asking-price level (₦/sqm) |
| Supporting Source | One supporting source/listing |
| Date Searched | Search date |


In [37]:
import pandas as pd

records = []
for town in towns:
    result = overview_cache.get(town) or {}
    text = result.get("text") or ""
    price = extract_price_ngn_per_sqm(text) if not result.get("error") else None
    source = first_source(result.get("references") or [])
    if not source and text:
        source = "AI Overview text only (no structured reference returned)"
    if result.get("error"):
        price = price or "ERROR"
        source = result["error"]
    elif not price:
        price = "Not found / needs manual review"

    records.append(
        {
            "Town": town,
            "Current Land Price/sqm": price,
            "Supporting Source": source,
            "Date Searched": search_date,
            "Query": result.get("query", QUESTION_TEMPLATE.format(town=town)),
            "Overview excerpt": (text[:280] + "...") if len(text) > 280 else text,
        }
    )

market_df = pd.DataFrame(records)
display_cols = ["Town", "Current Land Price/sqm", "Supporting Source", "Date Searched"]
market_df[display_cols]


,Town,Current Land Price/sqm,Supporting Source,Date Searched
0,Igando,₦85380/sqm,Property for sale!!!📢 We’re offering a plot of...,2026-09-25
1,Ikotun,₦140000/sqm,OWN A PIECE OF LAGOS PRESTIGE IN PG ESTATE ......,2026-09-25
2,Isheri-Olofin,₦196000/sqm,Distress Estate land for sale at Hotel Bus Sto...,2026-09-25
3,Abaranje / Okerube,₦60600/sqm,🎾👉* *_DISTRESS One Plot of Land Directly Facin...,2026-09-25
4,Egan,₦75000/sqm,"4+ Lands for Sale in Egan, Ikotun Igando, Lago...",2026-09-25


## 8. Inspect results (do not stop at run)

Check:

- Were all five towns processed?
- Did each town return a market price?
- Does the result look plausible for Lagos land markets?
- Was supporting evidence returned?
- Did anything fail or look unexpected?

Then experiment by changing `QUESTION_TEMPLATE` and re-running sections 5–7.


In [38]:
print("Towns processed:", len(market_df))
print("Price values:")
print(market_df["Current Land Price/sqm"].value_counts(dropna=False))
print("\nRows needing manual review:")
needs_review = market_df[
    market_df["Current Land Price/sqm"]
    .astype(str)
    .str.contains("Not found|ERROR|unclear|verify", case=False, na=False)
]
needs_review[display_cols] if len(needs_review) else "None flagged by heuristics."


Towns processed: 5
Price values:
Current Land Price/sqm
₦85380/sqm     1
₦140000/sqm    1
₦196000/sqm    1
₦60600/sqm     1
₦75000/sqm     1
Name: count, dtype: int64

Rows needing manual review:


'None flagged by heuristics.'

In [39]:
TOWN_TO_INSPECT = towns[3]
detail = overview_cache.get(TOWN_TO_INSPECT) or {}
print("Town:", TOWN_TO_INSPECT)
print("Has overview:", detail.get("has_ai_overview"))
print("Raw AI keys:", detail.get("raw_keys"))
print("\nFull text:\n")
print(detail.get("text") or "(empty)")
print("\nReferences:")
for i, ref in enumerate(detail.get("references") or [], start=1):
    if isinstance(ref, dict):
        print(f"{i}. {ref.get('title')} -> {ref.get('link')}")
    else:
        print(f"{i}. {ref}")


Town: Abaranje / Okerube
Has overview: True
Raw AI keys: ['references', 'text_blocks']

Full text:

The estimated land price in Abaranje / Okerube, Lagos State ranges from roughly ₦53,000 to ₦60,600 per square metre, based on current local listings for full plots.
Price Breakdown & Listings
Full Plot (660 sqm) on Ile Eja Road: Listed around ₦35,000,000 to ₦40,000,000 , which breaks down to approximately ₦53,030 – ₦60,606 per sqm.
Smaller / Subdivided Plots: Smaller fractional plots (such as quarter plots or 300sqm – 500sqm allocations in managed/gated extensions) are listed starting around ₦10,000,000 per fraction or with initial deposits near ₦5,000,000 .
Prices vary depending on the exact road accessibility (e.g., facing main tarred roads like Ile Eja Road versus inner streets), title documents available (C of O, Registered Survey, or Receipt and Deed), and overall negotiation.
If you'd like, I can help you with:
Estimating the total cost for a specific land size (e.g., 300sqm or 500

## 9. Export your market dataset

Download these files for Submission 2 (dataset + repo).


In [40]:
from pathlib import Path

out_dir = Path("sprint2_outputs")
out_dir.mkdir(exist_ok=True)

csv_path = out_dir / "market_dataset.csv"
json_path = out_dir / "market_dataset.json"
overviews_path = out_dir / "ai_overview_raw.json"

export_df = market_df[display_cols].copy()
export_df.to_csv(csv_path, index=False)
export_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

serialisable = {}
for town, payload in overview_cache.items():
    serialisable[town] = {
        "query": payload.get("query"),
        "error": payload.get("error"),
        "text": payload.get("text"),
        "references": payload.get("references"),
        "has_ai_overview": payload.get("has_ai_overview"),
    }
overviews_path.write_text(json.dumps(serialisable, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", csv_path.resolve())
print("Wrote:", json_path.resolve())
print("Wrote:", overviews_path.resolve())

try:
    from google.colab import files  # type: ignore
    files.download(str(csv_path))
    files.download(str(json_path))
except Exception:
    print("Not in Colab download mode; files are on disk in sprint2_outputs/.")


Wrote: /content/sprint2_outputs/market_dataset.csv
Wrote: /content/sprint2_outputs/market_dataset.json
Wrote: /content/sprint2_outputs/ai_overview_raw.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Optional experiments

1. Change the question wording and compare prices/sources
2. Add "residential" or "commercial" to the prompt
3. Test one extra nearby town (not required for submission)
4. If a town fails, use an AI assistant to interpret the error or overview text

Goal: understand what the technology can do for a spatial market question.


In [ ]:
# Example: re-run a single town with an alternate prompt
# alt_query = f"Residential land asking price per square metre in {towns[0]}, Lagos Nigeria"
# alt = fetch_ai_overview(alt_query)
# print(alt.get("text") or alt.get("error"))
# print(extract_price_ngn_per_sqm(alt.get("text") or ""))
